# run_scoring
Stage 4 canonical scoring CPU worker. Run cells 2, 3 and 4 in order. Configure only cell 3. Use the pinned CPU runtime and exact integrated Stage 4 commit.

- Set `MODE = 'model'` with `MODEL = 'qwen'` or `MODEL = 'gemma'` to run Qwen and Gemma on separate CPU runtimes (70 blocks each: 258 SCREEN routes and 4 E19 DEV-C diagonals).
- Set `MODE = 'e03'` with `MODEL = None` to run shared E03 once (13 SCREEN blocks: 12 language-specific diagonals and 1 pooled four-view character block).
- `SCOPE = 'all'` publishes each model's Priority 0 E01 midpoint SCREEN grid and E19 source DEV-C diagonals first and streams a `priority_handoff` event so E19/E25 can begin immediately. Use `SCOPE = 'profile'` to measure one deterministic metadata-selected representative block first and reuse its reference in `RESUME_REFS`/`RESUME_FILES`.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage4-scoring')
CODE_REF = 'phase1'  # Resolved once below; set this to the approved Stage 4 commit to pin explicitly.
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Drazr/Research.git', str(REPO)], check=True)
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git', 'fetch', 'origin', CODE_REF], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO, text=True).strip()
subprocess.run(['git', 'checkout', '--detach', CODE_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements-stage2-cpu.txt')], check=True)
print('Fixed code commit:', CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
STAGE4_PLAN = PHASE1_ROOT / 'REPLACE_STAGE4_PLAN/_COMPLETE.json'  # Path or explicit {'path': ..., 'sha256': ...} dict.
MODE = 'model'          # 'model' (Qwen/Gemma) or 'e03' (shared E03, runs once).
MODEL = 'qwen'          # 'qwen' or 'gemma' when MODE == 'model'; None when MODE == 'e03'.
SCOPE = 'all'           # 'all', 'priority' (models only), or 'profile'.
RESUME_REFS = []        # Optional explicit {'path': ..., 'sha256': ...} completed prediction refs for this owner.
RESUME_FILES = []       # Optional mounted JSON files containing completed prediction refs for this owner.
OUTPUT_INVENTORY = None # Optional mounted JSON path (e.g. PHASE1_ROOT / 'results/stage4/scoring_qwen.json').


In [ ]:
import json, signal
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Stage 4 checkout.')
if MODE not in ('model', 'e03'):
    raise RuntimeError("MODE must be 'model' or 'e03'.")
if MODE == 'model' and MODEL not in ('qwen', 'gemma'):
    raise RuntimeError("Set MODEL to 'qwen' or 'gemma' when MODE == 'model'.")
if MODE == 'e03' and MODEL is not None:
    raise RuntimeError("Set MODEL = None when MODE == 'e03'.")
if isinstance(STAGE4_PLAN, dict):
    manifest_arg = json.dumps(STAGE4_PLAN)
    if not (PHASE1_ROOT / STAGE4_PLAN['path']).is_file():
        raise RuntimeError(f"Set the explicit mounted Stage 4 plan in cell 3: {STAGE4_PLAN['path']}")
else:
    plan_path = Path(STAGE4_PLAN)
    if not plan_path.is_file():
        raise RuntimeError(f'Set the explicit mounted Stage 4 plan in cell 3: {plan_path}')
    manifest_arg = str(plan_path)
command = [
    sys.executable, '-u', '-m', 'src.stage4.run_scoring',
    '--phase1-root', str(PHASE1_ROOT),
    '--manifest', manifest_arg,
    '--mode', MODE,
    '--scope', SCOPE,
    '--storage-scope', 'persistent-drive',
]
if MODEL is not None:
    command += ['--model', MODEL]
for ref in RESUME_REFS:
    command += ['--resume-ref', json.dumps(ref) if isinstance(ref, dict) else str(ref)]
for resume_file in RESUME_FILES:
    rf_path = Path(resume_file)
    if not rf_path.is_file():
        raise RuntimeError(f'Missing explicit resume file: {rf_path}')
    command += ['--resume-file', str(rf_path)]
if OUTPUT_INVENTORY is not None:
    command += ['--output-inventory', str(OUTPUT_INVENTORY)]
process = subprocess.Popen(
    command, cwd=REPO, env=os.environ.copy(),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    raise
if process.wait() != 0:
    raise RuntimeError('Scoring worker blocked/failed. Preserve streamed references and inspect the error before resuming.')
